# Model Benchmarking
This notebook compares XGBoost, ECG ResNet, and Hybrid Ensemble predictions.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.load_dataset import load_or_create_datasets
from src.data.preprocessing import preprocess_ecg_data, preprocess_tabular_data
from src.models.cnn_ecg import ECGCNN
from src.models.xgboost_model import XGBoostModelWrapper
from src.utils.config import Config
from src.utils.metrics import evaluate_predictions

config = Config(str(PROJECT_ROOT / "configs/params.yaml"), str(PROJECT_ROOT / "configs/paths.yaml"))
sns.set_theme(style="whitegrid")

In [2]:
df, ecg = load_or_create_datasets(config)
X_train, X_test, y_train, y_test, _ = preprocess_tabular_data(df, config)
processed_ecg = preprocess_ecg_data(ecg, config)
X_ecg_test = processed_ecg[X_test.index.to_numpy()]
y_ecg_test = y_test.to_numpy()

xgb_wrapper = XGBoostModelWrapper(config)
xgb_wrapper.load(config.get_path("models")["xgboost_path"])
xgb_pred = xgb_wrapper.predict(X_test)
xgb_prob = xgb_wrapper.predict_proba(X_test)[:, 1]
xgb_metrics = evaluate_predictions(y_test, xgb_pred, xgb_prob)

cnn_model = ECGCNN(in_channels=processed_ecg.shape[1], num_classes=2)
cnn_model.load_state_dict(torch.load(config.get_path("models")["ecg_cnn_path"], map_location="cpu"))
cnn_model.eval()
with torch.no_grad():
    cnn_logits = cnn_model(torch.tensor(X_ecg_test, dtype=torch.float32))
cnn_pred = cnn_logits.argmax(dim=1).numpy()
cnn_prob = torch.softmax(cnn_logits, dim=1)[:, 1].numpy()
cnn_metrics = evaluate_predictions(y_ecg_test, cnn_pred, cnn_prob)

Tabular data file not found at data/raw/mimic_patients.csv. Generating mock data.
Saved mock tabular data to data/raw/mimic_patients.csv
ECG signals file not found at data/raw/ecg_signals.npy. Generating mock ECG dataset.
Saved mock ECG data to data/raw/ecg_signals.npy


FileNotFoundError: [Errno 2] No such file or directory: 'experiments/saved_models/xgboost_shap_model.pkl'

In [ ]:
metrics_table = pd.DataFrame([xgb_metrics, cnn_metrics], index=["XGBoost", "ECG CNN"])
metrics_table.drop(columns="confusion_matrix").round(4)

In [ ]:
output_dir = Path(config.get_path("outputs")["outputs_dir"])
output_dir.mkdir(parents=True, exist_ok=True)
metrics_table.drop(columns="confusion_matrix").to_csv(config.get_path("outputs")["metrics_csv_path"])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for axis, (name, metrics) in zip(axes, [("XGBoost", xgb_metrics), ("ECG CNN", cnn_metrics)]):
    cm = np.array([[metrics["confusion_matrix"]["tn"], metrics["confusion_matrix"]["fp"]],
                   [metrics["confusion_matrix"]["fn"], metrics["confusion_matrix"]["tp"]]])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axis)
    axis.set_title(f"{name} Confusion Matrix")
    axis.set_xlabel("Predicted")
    axis.set_ylabel("Actual")
plt.tight_layout()
plt.show()